# 08 · ephys cell summary: one row per cell

This notebook joins the final values of **03 membrane test, 04 baseline, 05 steps, 06 ramp and 07 timecourse** into
**one row per cell** (Cell Tag from the recording book). The cell is the unit of analysis: when a cell has several
recordings of one protocol, **one** is used, never an average.

**Which recording?** (1) an override in `config.yaml` if set; otherwise (2) the **first recording that passes QC**,
in recording order (earliest = closest to break-in and before any drug). Recordings with an `exclude_if_flag` flag
(e.g. a baseline recorded after a drug washout) are only used if nothing else is available. Every choice and its
alternatives are listed in `selection.csv`.

Run 01–07 first. A protocol whose output is missing is simply left empty.

In [ ]:
OVERWRITE = True

import pandas as pd
from IPython.display import display
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 250)

from ephys_analysis.cell_summary import build_summary, load_tables
from ephys_analysis.utils import RunLog, check_writable, find_repo_root, load_config, repo_path, stamp, write_csv

REPO = find_repo_root()
cfg = load_config(REPO / "08_ephys_cell-summary")
out = REPO / "08_ephys_cell-summary" / "output"
targets = [out / "cell_summary.csv", out / "selection.csv"]
check_writable(targets, OVERWRITE)
log = RunLog(cfg, out)

## Step 1 · Load the cells and the protocol outputs

In [ ]:
cells = pd.read_csv(repo_path(cfg, cfg["cells_table"]), dtype={"cell_tag": "string"})
tables = load_tables(cfg, repo_path)
display(pd.DataFrame({"protocol": list(tables), "recordings": [len(t) for t in tables.values()]}))
display(cells[["cell_tag", "n_recordings", "abf_families"]])

## Step 2 · Select one recording per cell and protocol
Review: is the chosen recording the right one? To force another, add it under `overrides:` in `config.yaml`
and re-run.

In [ ]:
summary, selection = build_summary(cells, tables, cfg, log)
display(selection[["cell_tag", "protocol", "abf_filename", "n_recordings", "reason", "alternatives"]])

## Step 3 · The cell summary

In [ ]:
display(summary.set_index("cell_tag").T)

## Step 4 · Save

In [ ]:
for df, p in zip((summary, selection), targets):
    write_csv(stamp(df, log.run_id), p)
    print("written:", p)
log.finish(targets)